# 03 · Feature engineering  *(heaviest)*
**Owner:** Deepaka · **Branch:** `data/03-features`

Build **summary** + **discharge-curve** features on top of `preprocessed.parquet` and the raw `bat_dict` curves. ~42 features; no leakage (feature at cycle n uses only cycles ≤ n + early-life ref; warm-up cycles ≤20 dropped). Saves `features.parquet` + `feature_list.json`. Near-duplicate removal happens in notebook 04.


In [ ]:
# --- Colab setup (run first) ---
import os
REPO = '/content/ev-battery-rul-prediction'   # absolute path -> safe to re-run
if not os.path.exists(REPO):
    !git clone https://github.com/Chathumi-Senanayake/ev-battery-rul-prediction.git {REPO}
%cd {REPO}
!git pull -q
!pip install -q -r requirements.txt
import sys
if REPO not in sys.path: sys.path.insert(0, REPO)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/ev_battery_data'
except Exception as e:
    print('Drive mount failed -> using local storage:', e)
    DATA_DIR = '/content/ev_battery_data'
os.makedirs(DATA_DIR, exist_ok=True)
print('DATA_DIR =', DATA_DIR)


In [ ]:
import pickle, json, numpy as np, pandas as pd
from src.features import build_features, feature_columns, META

clean = pd.read_parquet(os.path.join(DATA_DIR,'preprocessed.parquet'))
bat_dict = pickle.load(open(os.path.join(DATA_DIR,'bat_dict.pkl'),'rb'))
# keep only cells that survived preprocessing
keep = set(clean['cell'].unique())
bat_dict = {c:d for c,d in bat_dict.items() if c in keep}
print('cells:', len(bat_dict))


### Build features
`build_features` does both groups, merges on (cell, cycle), fills invalid curve cycles from the previous valid one, drops warm-up cycles, drops NaN.


In [ ]:
features = build_features(clean, bat_dict)
cols = feature_columns(features)
print('rows:', len(features), '| cells:', features['cell'].nunique(), '| features:', len(cols))
features[['cell','cycle','RUL'] + cols].head()


### Feature–RUL correlation (sanity) + save
Spearman with RUL per feature (higher |ρ| = more predictive). The actual near-duplicate filtering is done on training cells only in notebook 04.


In [ ]:
corr = features[cols].apply(lambda s: s.corr(features['RUL'], method='spearman'))
print(corr.abs().sort_values(ascending=False).round(3).to_string())


In [ ]:
features.to_parquet(os.path.join(DATA_DIR,'features.parquet'), index=False)
json.dump(cols, open(os.path.join(DATA_DIR,'feature_list.json'),'w'), indent=2)
print('saved features.parquet with', len(cols), 'features')
